In [ ]:
import zipfile
import os
import cv2
import numpy as np

zip_path = "/content/archive (21).zip"
extract_destination = "brain_tumor_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_destination)

print("Dataset Extracted Successfully")

# Count the number of extracted files/images
extracted_image_count = 0
for dirpath, dirnames, filenames in os.walk(extract_destination):
    extracted_image_count += len(filenames)

print("Total images loaded:", extracted_image_count)

Dataset Extracted Successfully
Total images loaded: 506


In [ ]:
import os
import cv2
import numpy as np

images = []
labels = []

dataset_path = "brain_tumor_dataset"

for label_name in ["yes", "no"]:

    folder_path = os.path.join(dataset_path, label_name)

    for file in os.listdir(folder_path):

        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path)

        if img is not None:

            # Resize
            img = cv2.resize(img, (128, 128))

            # Normalize
            img = img / 255.0

            images.append(img)

            if label_name == "yes":
                labels.append(1)
            else:
                labels.append(0)

X = np.array(images)
y = np.array(labels)

print("Images Shape:", X.shape)
print("Labels Shape:", y.shape)

Images Shape: (253, 128, 128, 3)
Labels Shape: (253,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Images:", X_train.shape)
print("Testing Images :", X_test.shape)

Training Images: (202, 128, 128, 3)
Testing Images : (51, 128, 128, 3)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()

# Convolution Layer 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

# Convolution Layer 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Convolution Layer 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten
model.add(Flatten())

# Dense Layers
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.5396 - loss: 0.7532 - val_accuracy: 0.6471 - val_loss: 0.6105
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 854ms/step - accuracy: 0.7277 - loss: 0.5855 - val_accuracy: 0.7647 - val_loss: 0.4698
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 839ms/step - accuracy: 0.7723 - loss: 0.5434 - val_accuracy: 0.7451 - val_loss: 0.5083
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 830ms/step - accuracy: 0.8119 - loss: 0.4844 - val_accuracy: 0.7451 - val_loss: 0.4801
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 848ms/step - accuracy: 0.7921 - loss: 0.4701 - val_accuracy: 0.7647 - val_loss: 0.4542
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 780ms/step - accuracy: 0.8069 - loss: 0.4601 - val_accuracy: 0.7451 - val_loss: 0.4754
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 814ms/step - accuracy: 0.8267 - loss: 0.3864 - val_accuracy: 0.7647 - val_loss: 0.4134
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 784ms/step - accuracy: 0.8267 - loss: 0.3637 - val_accuracy: 0.7843 - val_los

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.7843 - loss: 0.5167
Test Accuracy: 0.7843137383460999


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Predict probabilities
y_pred = model.predict(X_test)

# Convert probabilities to 0 or 1
y_pred = (y_pred > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 511ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.50      0.65        20
           1       0.75      0.97      0.85        31

    accuracy                           0.78        51
   macro avg       0.83      0.73      0.75        51
weighted avg       0.81      0.78      0.77        51

Confusion Matrix:
[[10 10]
 [ 1 30]]


In [ ]:
import cv2
import numpy as np

image_path = "/content/test_image.jpeg"

img = cv2.imread(image_path)
img = cv2.resize(img, (128,128))
img = img / 255.0

img = np.expand_dims(img, axis=0)

prediction = model.predict(img)

if prediction[0][0] > 0.5:
    print("Tumor Detected")
else:
    print("No Tumor Detected")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Tumor Detected


In [ ]:
import cv2
import numpy as np

image_path = "/content/test_image1.jpg"

img = cv2.imread(image_path)
img = cv2.resize(img, (128,128))
img = img / 255.0

img = np.expand_dims(img, axis=0)

prediction = model.predict(img)

if prediction[0][0] > 0.5:
    print("Tumor Detected")
else:
    print("No Tumor Detected")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
No Tumor Detected
